In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from openai import OpenAI
client = OpenAI()

In [3]:
def llm_chat(prompt):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [26]:
llm_chat("hello world")

'Hello world!'

In [27]:
question = "Should I go to the game tomorrow?"
answer = llm_chat(question)
print(answer)

Maybe — but it depends on a few practical things.

Ask yourself:
- **Do you actually want to go?**
- **Can you afford it** (ticket, food, transport, time)?
- **Will going help or hurt your plans** for tomorrow?
- **Would you regret missing it** if it turns out to be a great game?

If you’re on the fence, a simple rule:
- **Go** if you’re excited and it won’t create stress.
- **Skip it** if you’re tired, short on money, or it will disrupt something more important.

If you want, tell me **what game it is, where it is, and what’s holding you back**, and I can help you decide.


In [28]:
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm_chat(user_prompt)

In [4]:
import requests

url = 'https://datatalksclub.github.io/faq/json/courses.json'
response = requests.get(url)
courses_raw = response.json()

In [33]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 398},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249},
 {'course': 'production-ai-agents-generative-media',
  'course_name': 'Production AI Agents for Generative Media',
  'path': '/json/production-ai-agents-generative-media.json',
  'questions_count': 19},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 161},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 76},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'machine-learning-zoomcamp',
 

In [5]:
# Create FAQ knowledge base
documents = []
base_url = 'https://datatalksclub.github.io/faq'

for course in courses_raw:
    course_path = f'{base_url}{course['path']}'

    course_response = requests.get(course_path)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [6]:
len(documents)

1489

In [7]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [8]:
from minsearch import Index

index = Index(
    text_fields=['question', 'answer', 'section'],
    keyword_fields=['course']
)

index.fit(documents)

In [13]:
search_results = index.search(
    'I just discovered the course. Can I still join?',
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [14]:
def search(question, course='llm-zoomcamp'):
    return index.search(
        question,
        boost_dict={'question': 2.0, 'section': 0.5},
        filter_dict={'course': course},
        num_results=5
    )

question = 'I just discovered the course. Can I still join?'

search_results = search(question)

In [15]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer
is not found in the context, respond with "I don't know."
'''

In [17]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()


In [19]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expec

In [20]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

"\nQuestion:\nI just discovered the course. Can I still join?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: The homework submission form is still open even though the deadline has passed — can I still submit?\nA: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.\n\nGeneral Course-Related Questions\nQ: I missed the first homework - can I still get a certificate?\nA: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n\nGeneral Cours

In [21]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )
    return prompt.strip()

In [22]:
prompt = build_prompt(question, search_results)

In [23]:
print(prompt)

Question:
I just discovered the course. Can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions

In [24]:
def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    return llm_chat(prompt)

In [9]:
import importlib
import rag_helper

importlib.reload(rag_helper)

<module 'rag_helper' from '/workspaces/llm-zoomcamp/rag_helper.py'>

In [10]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.
